# 6. Results Summary & Discussion

Cross-store comparison of the three forecasting methods, and the ethical/social/policy discussion for the paper. Corresponds to step 6 of the workflow in `CLAUDE.md`.

In [ ]:
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from plotting import set_paper_style, comma_axis, dataframe_to_image, METHOD_COLORS, METHOD_LABELS, METHOD_ORDER, INK_PRIMARY

set_paper_style()

OUTPUTS = Path.cwd().parent / "outputs"
FIGURES = OUTPUTS / "figures"
results = pd.read_csv(OUTPUTS / "tables" / "model_comparison.csv")
store_meta = pd.read_csv(OUTPUTS / "tables" / "store_selection.csv")

test_results = (
    results.query("split == 'test'")
    .merge(store_meta, left_on="store", right_on="Store")
)
test_results

## Forecast accuracy by store and method (RMSE and MAE)

Both metrics from Section 2.6 are reported, each as a bar chart and a table: RMSE weights large misses more heavily, while MAE gives the typical error size in directly interpretable sales units.

In [ ]:
stores = sorted(test_results["store"].unique())
x = np.arange(len(stores))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 4))
for i, method in enumerate(METHOD_ORDER):
    vals = [
        test_results.query("store == @s and method == @method")["rmse"].iloc[0]
        for s in stores
    ]
    bars = ax.bar(x + (i - 1) * width, vals, width, label=METHOD_LABELS[method], color=METHOD_COLORS[method])
    ax.bar_label(bars, fmt="%.0f", padding=2, fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels([f"Store {s}" for s in stores])
comma_axis(ax, "y")
ax.set_ylabel("Test RMSE (sales)")
ax.set_title("Forecast error by method and store")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUTPUTS / "figures" / "rmse_comparison.png")
plt.show()

In [ ]:
rmse_table = (
    test_results.pivot(index="store", columns="method", values="rmse")
    .rename(columns=METHOD_LABELS)[[METHOD_LABELS[m] for m in METHOD_ORDER]]
    .round(0)
    .astype(int)
    .reset_index()
    .rename(columns={"store": "Store"})
)
dataframe_to_image(rmse_table, FIGURES / "rmse_table.png", title="Test RMSE by store and method")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for i, method in enumerate(METHOD_ORDER):
    vals = [
        test_results.query("store == @s and method == @method")["mae"].iloc[0]
        for s in stores
    ]
    bars = ax.bar(x + (i - 1) * width, vals, width, label=METHOD_LABELS[method], color=METHOD_COLORS[method])
    ax.bar_label(bars, fmt="%.0f", padding=2, fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels([f"Store {s}" for s in stores])
comma_axis(ax, "y")
ax.set_ylabel("Test MAE (sales)")
ax.set_title("Forecast error (MAE) by method and store")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES / "mae_comparison.png")
plt.show()

In [ ]:
mae_table = (
    test_results.pivot(index="store", columns="method", values="mae")
    .rename(columns=METHOD_LABELS)[[METHOD_LABELS[m] for m in METHOD_ORDER]]
    .round(0)
    .astype(int)
    .reset_index()
    .rename(columns={"store": "Store"})
)
dataframe_to_image(mae_table, FIGURES / "mae_table.png", title="Test MAE by store and method")

## Best method per store

In [ ]:
best = (
    test_results.sort_values("rmse")
    .groupby("store")
    .first()
    [["method", "rmse", "mae", "StoreType", "Promo2", "DistanceTier"]]
)
best

In [ ]:
paper_best = best.reset_index().rename(columns={"store": "Store"})
paper_best["method"] = paper_best["method"].map(METHOD_LABELS)
paper_best["rmse"] = paper_best["rmse"].round(0).astype(int)
paper_best["mae"] = paper_best["mae"].round(0).astype(int)
paper_best.columns = ["Store", "Best method", "RMSE", "MAE", "Store type", "Promo2", "Distance tier"]
dataframe_to_image(paper_best, FIGURES / "best_method_table.png", title="Best-performing method per store")

## Relative error (scaled to the seasonal-naive baseline)

Raw RMSE/MAE are not directly comparable across stores of different sales scale (see Discussion). Dividing each method's error by the seasonal-naive baseline's error *for the same store* removes that scale dependence - a value of 0.50 means that method's error was half the size of the naive baseline's, regardless of the store's absolute sales volume, and is directly comparable across stores. This is the same idea behind MASE (Mean Absolute Scaled Error). The seasonal-naive baseline itself is omitted below, since by construction it is always 1.00 against itself for both metrics.

In [ ]:
naive = test_results.query("method == 'seasonal_naive'")[["store", "rmse", "mae"]].rename(
    columns={"rmse": "naive_rmse", "mae": "naive_mae"}
)
relative = test_results.merge(naive, on="store")
relative["relative_rmse"] = relative["rmse"] / relative["naive_rmse"]
relative["relative_mae"] = relative["mae"] / relative["naive_mae"]

combined = relative.pivot(index="store", columns="method", values=["relative_rmse", "relative_mae"])
combined = combined[[
    ("relative_rmse", "sarima"), ("relative_mae", "sarima"),
    ("relative_rmse", "lag_ml"), ("relative_mae", "lag_ml"),
]]
combined.columns = [
    f"{METHOD_LABELS['sarima']} (RMSE)", f"{METHOD_LABELS['sarima']} (MAE)",
    f"{METHOD_LABELS['lag_ml']} (RMSE)", f"{METHOD_LABELS['lag_ml']} (MAE)",
]
combined = combined.map(lambda v: f"{v:.2f}").reset_index().rename(columns={"store": "Store"})
dataframe_to_image(
    combined, FIGURES / "relative_error_table.png",
    title="Error relative to seasonal-naive baseline (1.00 = no improvement)",
)
combined

## Example forecast: actual vs. predicted

A concrete illustration of what the RMSE/MAE numbers above mean in practice: the real sales for one store's test window, alongside what each method actually predicted. Change `example_store` below to look at a different store.

In [ ]:
predictions = pd.read_csv(OUTPUTS / "tables" / "test_predictions.csv")

example_store = 26  # change to any store id from store_meta["Store"] to compare
example = predictions.query("store == @example_store")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(example["test_day"], example["Actual"], color=INK_PRIMARY, linewidth=2, marker="o", markersize=3, label="Actual", zorder=3)
for method in METHOD_ORDER:
    ax.plot(example["test_day"], example[method], color=METHOD_COLORS[method], linewidth=1.5, marker="o", markersize=3, label=METHOD_LABELS[method])

comma_axis(ax, "y")
ax.set_xlabel("Days into the test window (last ~6 weeks)")
ax.set_ylabel("Sales")
ax.set_title(f"Store {example_store}: actual vs. predicted sales over the test window")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES / f"example_forecast_store_{example_store}.png")
plt.show()

## Discussion: ethical, social, and policy implications

Store-level sales forecasts like these are typically consumed downstream by staffing and inventory-ordering systems: a store predicted to have low demand next week may be scheduled fewer staff-hours, and a store predicted to have high demand may receive priority stock allocation. This creates a direct link between forecast *error* and real workplace outcomes - a store whose sales the model under-predicts risks being understaffed relative to actual footfall (worse customer service, more pressure on whoever is on shift), while over-prediction ties up inventory capital and can lead to wasted stock (particularly relevant for perishable assortments).

The `best`-method table above shows whether forecast accuracy is uniform across store characteristics or varies systematically with StoreType, Promo2, or CompetitionDistance - fill in the actual pattern found here once the notebook has been run. If a retailer used a single shared model (or a single method) across all stores, any such unevenness would mean some store types - and by extension the workers and communities they serve - systematically receive worse-calibrated staffing and stock decisions than others. That is a fairness question distinct from raw average accuracy: even a model with good aggregate performance can disadvantage a subset of stores (e.g. smaller or more rural locations, or a StoreType with sparser training data) in a way that compounds over time.

A responsible deployment would monitor forecast error *by store segment*, not just in aggregate, and treat a persistently under-served segment as a signal to revisit the model - or fall back to a simpler, more conservative baseline for those stores - rather than silently accepting worse service for them.